# 3-1. 재순위화 (Reranking)

⏱ **소요시간**: 25분

## 학습 목표
- Bi-encoder와 Cross-encoder의 구조·속도·품질 차이를 설명할 수 있다.
- BGE-reranker-v2-m3(🔒)를 사용해 기존 retriever의 상위 K 결과를 재정렬할 수 있다.
- **2-stage retrieval** 패턴(dense top-100 → cross-encoder top-10)을 구현할 수 있다.
- S2에서 만든 캡스톤 5개 질문에 대해 reranker 적용 전후 결과를 비교·평가할 수 있다.

## 핵심 키워드
Bi-encoder · Cross-encoder · BGE-reranker-v2-m3 · Cohere Rerank · 2-stage retrieval · NDCG

## 사전 준비
- S2에서 이미 Chroma 벡터스토어와 전자금융 표준약관 문서를 구축했다고 가정한다.
- 이 노트북은 같은 코퍼스(`data/corpus_ko.txt`)를 사용해 자체 완결형으로도 실행할 수 있다.


In [ ]:
# 공통 세팅: common 헬퍼를 사용하기 위해 repo 루트를 sys.path에 추가
import sys, os
sys.path.insert(0, '../..')

from common import get_llm, get_embeddings, provider_badge
from common.ko_tokenizer import tokenize_ko
from common.loaders import load_any

print(provider_badge())


## 1. 왜 재순위화가 필요한가?

### Bi-encoder (🔒 빠름, 품질 중간)
```
질문 ─▶ [Encoder] ─▶ q_vec ┐
                              ├─ cosine ─▶ score
문서 ─▶ [Encoder] ─▶ d_vec ┘
```
- 문서 임베딩을 **오프라인에서 미리** 계산해 저장 → 대규모 검색에 유리.
- 단점: 질문과 문서를 독립적으로 인코딩하므로 **상호작용 정보**가 없다.

### Cross-encoder (🔒 느림, 품질 높음)
```
[CLS] 질문 [SEP] 문서 [SEP] ─▶ [Encoder] ─▶ 관련성 score (스칼라)
```
- 질문과 문서를 **함께** 인코딩 → 토큰 간 attention으로 세밀한 의미 매칭.
- 단점: 모든 후보 문서에 대해 LLM forward를 1번씩 돌려야 해서 비용이 O(K).

### 실무 해결: 2-stage retrieval

| 단계 | 방식 | 후보 수 | 속도 | 역할 |
|------|------|--------|------|------|
| 1단계 | Bi-encoder (dense) | top-100 | 빠름 | recall 확보 |
| 2단계 | Cross-encoder (rerank) | top-10 | 느림 | precision 확보 |

> 💡 **직관**: 1단계는 "관련 있어 보이는 100개를 빠르게 긁어오고", 2단계는 "그중 정말 관련 있는 10개를 꼼꼼히 고른다".


## 2. 실습용 데이터·벡터스토어 준비

S2에서 쓴 파이프라인을 요약해 재구성한다 (`data/corpus_ko.txt` 사용).

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

# 코퍼스 로드 및 청킹
docs = load_any('../../data/corpus_ko.txt')
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(docs)
print(f'전체 청크 수: {len(chunks)}')

# 1단계 검색용 벡터스토어 (bi-encoder 기반)
vectordb = FAISS.from_documents(chunks, get_embeddings())
base_retriever = vectordb.as_retriever(search_kwargs={'k': 20})
print('FAISS 벡터스토어 구축 완료')


## 3. BGE-reranker-v2-m3 로드 🔒

[BAAI/bge-reranker-v2-m3](https://huggingface.co/BAAI/bge-reranker-v2-m3)는 한국어·영어·중국어를 포함한 100+개 언어를 지원하는 cross-encoder이다.

> ⚠️ 최초 실행 시 약 2.3GB 모델을 다운로드한다. 폐쇄망에서는 `HF_HOME`에 사전 반입이 필요하다.


In [ ]:
from FlagEmbedding import FlagReranker

# normalize=True로 설정하면 0~1 사이 확률 유사 점수로 변환
reranker = FlagReranker('BAAI/bge-reranker-v2-m3', use_fp16=True)

# 간단 테스트: 질문-문서 페어에 대한 점수
pairs = [
    ('망분리란 무엇인가?', '망분리는 업무망과 인터넷망을 분리해 보안을 강화한다.'),
    ('망분리란 무엇인가?', 'LLM은 Transformer 아키텍처 기반의 모델이다.'),
]
scores = reranker.compute_score(pairs, normalize=True)
for (q, d), s in zip(pairs, scores):
    print(f'score={s:.4f} | {d[:40]}...')


## 4. LangChain ContextualCompressionRetriever에 reranker 연결

LangChain의 `BaseDocumentCompressor`를 상속해 우리 reranker를 래핑하자.

In [ ]:
from typing import Sequence
from langchain_core.documents import Document
from langchain_core.callbacks import Callbacks
from langchain.retrievers import ContextualCompressionRetriever
from langchain_core.documents.compressor import BaseDocumentCompressor

class BgeRerankCompressor(BaseDocumentCompressor):
    '''BGE reranker를 LangChain compressor 인터페이스로 감싼 클래스.'''
    model_name: str = 'BAAI/bge-reranker-v2-m3'
    top_n: int = 5

    class Config:
        arbitrary_types_allowed = True

    def compress_documents(
        self,
        documents: Sequence[Document],
        query: str,
        callbacks: Callbacks = None,
    ) -> Sequence[Document]:
        if not documents:
            return []
        pairs = [(query, d.page_content) for d in documents]
        scores = reranker.compute_score(pairs, normalize=True)
        ranked = sorted(zip(documents, scores), key=lambda x: x[1], reverse=True)
        top = ranked[: self.top_n]
        # 점수를 메타데이터에 기록
        out = []
        for d, s in top:
            d.metadata['rerank_score'] = float(s)
            out.append(d)
        return out

compressor = BgeRerankCompressor(top_n=5)
rerank_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever,
)
print('2-stage retriever 준비 완료 (dense top-20 → rerank top-5)')


## 5. S2 캡스톤 5개 질문으로 전후 비교

S2에서 정의한 전자금융 표준약관 벤치마크 질문이다. 여기서는 `data/corpus_ko.txt`의 관련 문장을 정답(gold) 스니펫으로 간주한다.

In [ ]:
BENCHMARK = [
    {
        'q': '전자금융거래의 정의는 무엇인가?',
        'gold_keywords': ['전자적 장치', '자동화된 방식', '금융회사'],
    },
    {
        'q': '개인정보보호법에서 개인정보는 어떻게 정의되는가?',
        'gold_keywords': ['살아 있는 개인', '성명', '주민등록번호'],
    },
    {
        'q': '금융회사가 신용정보를 처리할 때 준수해야 하는 원칙은?',
        'gold_keywords': ['신용정보의 이용 및 보호', '최소한의 범위', '수탁자'],
    },
    {
        'q': '망분리란 무엇이며 왜 금융권에서 중요한가?',
        'gold_keywords': ['업무망', '인터넷망', '전자금융감독규정'],
    },
    {
        'q': 'RAG가 LLM 환각 문제에 어떻게 도움이 되는가?',
        'gold_keywords': ['검색', '컨텍스트', '환각'],
    },
]

def hit_rate(doc_texts: list[str], keywords: list[str]) -> float:
    '''상위 검색 결과에서 정답 키워드가 포함된 비율.'''
    joined = ' '.join(doc_texts)
    return sum(1 for k in keywords if k in joined) / len(keywords)


In [ ]:
import pandas as pd

rows = []
for item in BENCHMARK:
    q, kws = item['q'], item['gold_keywords']
    # 재순위화 없음 (top-5만 잘라 비교)
    base_docs = base_retriever.invoke(q)[:5]
    base_hit = hit_rate([d.page_content for d in base_docs], kws)
    # 재순위화 적용
    rerank_docs = rerank_retriever.invoke(q)
    rerank_hit = hit_rate([d.page_content for d in rerank_docs], kws)
    rows.append({
        '질문': q,
        'dense top-5 hit': f'{base_hit:.2%}',
        'rerank top-5 hit': f'{rerank_hit:.2%}',
        '개선': f'{(rerank_hit - base_hit):+.2%}',
    })

df = pd.DataFrame(rows)
df


## 6. 참고: Cohere Rerank API ☁️

해외망 접근이 가능한 환경이라면 Cohere의 매니지드 reranker를 쓸 수도 있다. 아래는 **실행하지 않는** 참고 코드이다.

```python
# ☁️ Cohere Rerank (airgap 환경에서는 사용 불가)
# pip install langchain-cohere
# from langchain_cohere import CohereRerank
# cohere_compressor = CohereRerank(model='rerank-multilingual-v3.0', top_n=5)
# cohere_retriever = ContextualCompressionRetriever(
#     base_compressor=cohere_compressor,
#     base_retriever=base_retriever,
# )
```

| 항목 | BGE-reranker-v2-m3 🔒 | Cohere Rerank ☁️ |
|------|----------------------|-----------------|
| 인프라 | GPU/CPU 로컬 | REST API |
| 비용 | 모델 다운로드 1회 | $1 / 1K queries |
| 한국어 품질 | 우수 (다국어 학습) | 우수 |
| 폐쇄망 가능 | ✅ | ❌ |
| 지연 | ~100ms (GPU) | ~200ms + 네트워크 |
